# CS224N 作业 1：探索词向量（25 分）
### <font color='blue'>截止时间：2026 年 1 月 13 日（星期二）下午 4:30</font>

欢迎来到 CS224N！

开始之前，请确保你已经**阅读了与本 notebook 位于同一目录下的 README.md**，其中包含重要的环境配置说明。你需要先安装一些 Python 库，才能顺利完成这份作业。本 notebook 提供了大量代码，我们也非常鼓励你在学习过程中阅读并理解这些代码 :)

如果你还不太熟悉 Python、Numpy 或 Matplotlib，我们建议你参加周五的复习课。该课程会被录制，相关材料也会发布在我们的[网站](http://web.stanford.edu/class/cs224n/index.html#schedule)上。CS231N 的 Python/Numpy [教程](https://cs231n.github.io/python-numpy-tutorial/)也是一个很好的资源。


**作业说明：** 请确保在完成过程中随时保存 notebook。提交说明位于 notebook 底部。

In [11]:
import os
import requests

# 设置代理环境变量（注意大小写，通常建议大写）
os.environ["HTTP_PROXY"] = "http://127.0.0.1:10808"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:10808"
os.environ["http_proxy"] = "http://127.0.0.1:10808"
os.environ["https_proxy"] = "http://127.0.0.1:10808"
os.environ["CURL_CA_BUNDLE"] = ""

In [13]:
# All Import Statements Defined Here
# Note: Do not add to this list.
# ----------------

import sys
assert sys.version_info[0] == 3
assert sys.version_info[1] >= 8

from platform import python_version
assert int(python_version().split(".")[1]) >= 5, "Please upgrade your Python version following the instructions in \
    the README.md file found in the same directory as this notebook. Your Python version is " + python_version()

from gensim.models import KeyedVectors
from gensim.test.utils import datapath
import pprint
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]

from datasets import load_dataset
imdb_dataset = load_dataset("stanfordnlp/imdb", name="plain_text")

import re
import numpy as np
import random
import scipy as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA

START_TOKEN = '<START>'
END_TOKEN = '<END>'
NUM_SAMPLES = 150

np.random.seed(0)
random.seed(0)
# ----------------

ConnectError: [WinError 10054] 远程主机强迫关闭了一个现有的连接。

## 词向量

词向量经常被用作下游 NLP 任务的基础组件，例如问答、文本生成、翻译等。因此，建立对词向量优点和局限性的直觉非常重要。在这里，你将探索两类词向量：一种来自*共现矩阵*，另一种来自 *GloVe*。

**术语说明：** “word vectors”和“word embeddings”这两个术语经常可以互换使用。“embedding”指的是我们将单词含义的某些方面编码到一个较低维空间中。正如 [Wikipedia](https://en.wikipedia.org/wiki/Word_embedding) 所说，*从概念上讲，它涉及一种数学嵌入：从每个单词占一个维度的空间，映射到维度低得多的连续向量空间*。

## 第 1 部分：基于计数的词向量（10 分）

大多数词向量模型都从下面这个想法出发：

*You shall know a word by the company it keeps（你可以通过一个词所处的上下文来认识它）（[Firth, J. R. 1957:11](https://en.wikipedia.org/wiki/John_Rupert_Firth)）*

许多词向量实现都基于这样一种想法：相似的词，也就是（近）同义词，会出现在相似的上下文中。因此，相似的词通常会与一组共享的词一起被说出或写出，也就是共享上下文。通过考察这些上下文，我们可以尝试为词构建嵌入。基于这个直觉，许多“传统”的词向量构造方法依赖词频计数。这里我们会详细介绍其中一种策略：*共现矩阵*（更多信息见[这里](https://web.stanford.edu/~jurafsky/slp3/6.pdf)或[这里](https://web.archive.org/web/20190530091127/https://medium.com/data-science-group-iitr/word-embedding-2d05d270b285)）。

### 共现

共现矩阵统计的是某些对象在某个环境中共同出现的频率。给定文档中出现的某个词 $w_i$，我们考虑围绕 $w_i$ 的*上下文窗口*。假设固定窗口大小为 $n$，那么这个窗口就是该文档中 $w_i$ 前面的 $n$ 个词和后面的 $n$ 个词，即 $w_{i-n} \dots w_{i-1}$ 和 $w_{i+1} \dots w_{i+n}$。我们构建一个*共现矩阵* $M$，它是一个对称的词-词矩阵，其中 $M_{ij}$ 表示在所有文档中，$w_j$ 出现在 $w_i$ 窗口内的次数。

**示例：固定窗口 n=1 的共现矩阵**：

文档 1："all that glitters is not gold"

文档 2："all is well that ends well"


|     *    | `<START>` | all | that | glitters | is   | not  | gold  | well | ends | `<END>` |
|----------|-------|-----|------|----------|------|------|-------|------|------|-----|
| `<START>`    | 0     | 2   | 0    | 0        | 0    | 0    | 0     | 0    | 0    | 0   |
| all      | 2     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| that     | 0     | 1   | 0    | 1        | 0    | 0    | 0     | 1    | 1    | 0   |
| glitters | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| is       | 0     | 1   | 0    | 1        | 0    | 1    | 0     | 1    | 0    | 0   |
| not      | 0     | 0   | 0    | 0        | 1    | 0    | 1     | 0    | 0    | 0   |
| gold     | 0     | 0   | 0    | 0        | 0    | 1    | 0     | 0    | 0    | 1   |
| well     | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 1    | 1   |
| ends     | 0     | 0   | 1    | 0        | 0    | 0    | 0     | 1    | 0    | 0   |
| `<END>`      | 0     | 0   | 0    | 0        | 0    | 0    | 1     | 1    | 0    | 0   |

在 NLP 中，我们通常使用 `<START>` 和 `<END>` 标记句子、段落或文档的开始与结束。这些标记会被纳入共现计数，用来包裹每个文档，例如：“`<START>` All that glitters is not gold `<END>`”。

矩阵的行（或列）可以根据词与词之间的共现提供词向量，但这些向量可能非常大。为了降低维度，我们采用奇异值分解（Singular Value Decomposition, SVD），它类似于 PCA，会选择前 $k$ 个主成分。SVD 过程会将共现矩阵 $A$ 分解为对角矩阵 $S$ 中的奇异值，以及 $U_k$ 中新的、更短的词向量。

这种降维会保留语义关系；例如，*doctor* 和 *hospital* 会比 *doctor* 和 *dog* 更接近。

如果你不熟悉特征值和 SVD，可以阅读[这里](https://davetang.org/file/Singular_Value_Decomposition_Tutorial.pdf)面向初学者的 SVD 入门。关于这些算法的更深入理解，还可以参考 CS168 的第 [7](https://web.stanford.edu/class/cs168/l/l7.pdf)、[8](http://theory.stanford.edu/~tim/s15/l/l8.pdf) 和 [9](https://web.stanford.edu/class/cs168/l/l9.pdf) 讲，其中提供了高层次讲解。在实际实现中，建议使用 numpy、scipy 或 sklearn 等 Python 包中已经写好的函数。虽然对大型语料进行完整 SVD 可能非常占内存，但也存在可扩展的方法，例如 Truncated SVD，可以高效提取前 $k$ 个向量成分。

### 绘制共现词嵌入

这里我们将使用 Large Movie Review Dataset。这是一个用于二分类情感分类的数据集，包含的数据量显著多于此前的基准数据集。我们提供了 25,000 条两极分化的电影评论用于训练，以及 25,000 条用于测试。除此之外，也有额外的未标注数据可供使用。下面我们提供了一个 `read_corpus` 函数，它会从数据集中抽取电影评论文本。该函数还会为每个文档添加 `<START>` 和 `<END>` 标记，并将单词转换为小写。你**不需要**进行任何其它预处理。

In [ ]:
def read_corpus():
    """ Read files from the Large Movie Review Dataset.
        Params:
            category (string): category name
        Return:
            list of lists, with words from each of the processed files
    """
    files = imdb_dataset["train"]["text"][:NUM_SAMPLES]
    return [[START_TOKEN] + [re.sub(r'[^\w]', '', w.lower()) for w in f.split(" ")] + [END_TOKEN] for f in files]

让我们看看这些文档是什么样子的……

In [ ]:
imdb_corpus = read_corpus()
pprint.pprint(imdb_corpus[:3], compact=True, width=100)
print("corpus size: ", len(imdb_corpus[0]))

### 问题 1.1：实现 `distinct_words` [代码]（2 分）

编写一个方法，用来找出语料库中出现过的不同单词（词类型）。

你可以使用 `for` 循环来处理输入 `corpus`（一个由字符串列表组成的列表），但也可以尝试使用 Python 列表推导式（通常更快）。特别是，[这篇内容](https://coderwall.com/p/rcmaea/flatten-a-list-of-lists-in-one-line-in-python)可能有助于将列表的列表展平成一个列表。如果你一般不熟悉 Python 列表推导式，可以参考[更多信息](https://python-3-patterns-idioms-test.readthedocs.io/en/latest/Comprehensions.html)。

你返回的 `corpus_words` 应该是排序后的。你可以使用 Python 的 `sorted` 函数来完成这一点。

你可能会发现使用 [Python set](https://www.w3schools.com/python/python_sets.asp) 去除重复单词很有用。

In [ ]:
def distinct_words(corpus):
    """ Determine a list of distinct words for the corpus.
        Params:
            corpus (list of list of strings): corpus of documents
        Return:
            corpus_words (list of strings): sorted list of distinct words across the corpus
            n_corpus_words (integer): number of distinct words across the corpus
    """
    corpus_words = []
    n_corpus_words = -1
    
    # ------------------
    # Write your implementation here.
    read_train_data = read_corpus()
    [corpus_words.append(word) for document in read_train_data for word in document if word not in corpus_words]
    corpus_words.sort()
    n_corpus_words = len(corpus_words)
    # -----------------

    return corpus_words, n_corpus_words

In [ ]:
# ---------------------
# Run this sanity check
# Note that this not an exhaustive check for correctness.
# ---------------------

# Define toy corpus
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
test_corpus_words, num_corpus_words = distinct_words(test_corpus)

# Correct answers
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
ans_num_corpus_words = len(ans_test_corpus_words)

# Test correct number of words
assert(num_corpus_words == ans_num_corpus_words), "Incorrect number of distinct words. Correct: {}. Yours: {}".format(ans_num_corpus_words, num_corpus_words)

# Test correct words
assert (test_corpus_words == ans_test_corpus_words), "Incorrect corpus_words.\nCorrect: {}\nYours:   {}".format(str(ans_test_corpus_words), str(test_corpus_words))

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### 问题 1.2：实现 `compute_co_occurrence_matrix` [代码]（3 分）

编写一个方法，为某个窗口大小 $n$（默认值为 4）构造共现矩阵，考虑中心词前面的 $n$ 个词和后面的 $n$ 个词。在这里，我们开始使用 `numpy (np)` 来表示向量、矩阵和张量。如果你不熟悉 NumPy，cs231n 的 [Python NumPy 教程](http://cs231n.github.io/python-numpy-tutorial/)后半部分提供了一个 NumPy 教程。


In [ ]:
def compute_co_occurrence_matrix(corpus, window_size=4):
    """ Compute co-occurrence matrix for the given corpus and window_size (default of 4).
    
        Note: Each word in a document should be at the center of a window. Words near edges will have a smaller
              number of co-occurring words.
              
              For example, if we take the document "<START> All that glitters is not gold <END>" with window size of 4,
              "All" will co-occur with "<START>", "that", "glitters", "is", and "not".
    
        Params:
            corpus (list of list of strings): corpus of documents
            window_size (int): size of context window
        Return:
            M (a symmetric numpy matrix of shape (number of unique words in the corpus , number of unique words in the corpus)): 
                Co-occurence matrix of word counts. 
                The ordering of the words in the rows/columns should be the same as the ordering of the words given by the distinct_words function.
            word2ind (dict): dictionary that maps word to index (i.e. row/column number) for matrix M.
    """
    words, n_words = distinct_words(corpus)
    M = None
    word2ind = {}
    
    # ------------------
    # Write your implementation here.
    
    
    # ------------------

    return M, word2ind

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness.
# ---------------------

# Define toy corpus and get student's co-occurrence matrix
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)

# Correct M and word2ind
M_test_ans = np.array( 
    [[0., 0., 0., 0., 0., 0., 1., 0., 0., 1.,],
     [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 1., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 0., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.,],
     [1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,],
     [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.,],
     [0., 0., 1., 0., 1., 1., 0., 0., 0., 1.,],
     [1., 0., 0., 1., 1., 0., 0., 0., 1., 0.,]]
)
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
word2ind_ans = dict(zip(ans_test_corpus_words, range(len(ans_test_corpus_words))))

# Test correct word2ind
assert (word2ind_ans == word2ind_test), "Your word2ind is incorrect:\nCorrect: {}\nYours: {}".format(word2ind_ans, word2ind_test)

# Test correct M shape
assert (M_test.shape == M_test_ans.shape), "M matrix has incorrect shape.\nCorrect: {}\nYours: {}".format(M_test.shape, M_test_ans.shape)

# Test correct M values
for w1 in word2ind_ans.keys():
    idx1 = word2ind_ans[w1]
    for w2 in word2ind_ans.keys():
        idx2 = word2ind_ans[w2]
        student = M_test[idx1, idx2]
        correct = M_test_ans[idx1, idx2]
        if student != correct:
            print("Correct M:")
            print(M_test_ans)
            print("Your M: ")
            print(M_test)
            raise AssertionError("Incorrect count at index ({}, {})=({}, {}) in matrix M. Yours has {} but should have {}.".format(idx1, idx2, w1, w2, student, correct))

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### 问题 1.3：实现 `reduce_to_k_dim` [代码]（1 分）

构造一个方法，对矩阵执行降维，生成 k 维嵌入。使用 SVD 取前 k 个成分，并生成一个新的 k 维嵌入矩阵。

**注意：** numpy、scipy 和 scikit-learn（`sklearn`）都提供了某种 SVD 实现，但只有 scipy 和 sklearn 提供 Truncated SVD 的实现，并且只有 sklearn 提供了用于计算大规模 Truncated SVD 的高效随机化算法。因此请使用 [sklearn.decomposition.TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)。

In [ ]:
def reduce_to_k_dim(M, k=2):
    """ Reduce a co-occurence count matrix of dimensionality (num_corpus_words, num_corpus_words)
        to a matrix of dimensionality (num_corpus_words, k) using the following SVD function from Scikit-Learn:
            - http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html
    
        Params:
            M (numpy matrix of shape (number of unique words in the corpus , number of unique words in the corpus)): co-occurence matrix of word counts
            k (int): embedding size of each word after dimension reduction
        Return:
            M_reduced (numpy matrix of shape (number of corpus words, k)): matrix of k-dimensioal word embeddings.
                    In terms of the SVD from math class, this actually returns U * S
    """    
    n_iters = 10    # Use this parameter in your call to `TruncatedSVD`
    M_reduced = None
    print("Running Truncated SVD over %i words..." % (M.shape[0]))
    
    # ------------------
    # Write your implementation here.
    
    
    # ------------------

    print("Done.")
    return M_reduced

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness 
# In fact we only check that your M_reduced has the right dimensions.
# ---------------------

# Define toy corpus and run student code
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)
M_test_reduced = reduce_to_k_dim(M_test, k=2)

# Test proper dimensions
assert (M_test_reduced.shape[0] == 10), "M_reduced has {} rows; should have {}".format(M_test_reduced.shape[0], 10)
assert (M_test_reduced.shape[1] == 2), "M_reduced has {} columns; should have {}".format(M_test_reduced.shape[1], 2)

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### 问题 1.4：实现 `plot_embeddings` [代码]（1 分）

这里你将编写一个函数，在二维空间中绘制一组二维向量。对于图表，我们会使用 Matplotlib（`plt`）。

在这个例子中，你可能会发现改编[这段代码](http://web.archive.org/web/20190924160434/https://www.pythonmembers.club/2018/05/08/matplotlib-scatter-plot-annotate-set-text-at-label-each-point/)很有用。以后如果想画图，一个好方法是查看 [Matplotlib 图库](https://matplotlib.org/gallery/index.html)，找到一个看起来和你想要的图相似的示例，然后改编他们给出的代码。

In [ ]:
def plot_embeddings(M_reduced, word2ind, words):
    """ Plot in a scatterplot the embeddings of the words specified in the list "words".
        NOTE: do not plot all the words listed in M_reduced / word2ind.
        Include a label next to each point.
        
        Params:
            M_reduced (numpy matrix of shape (number of unique words in the corpus , 2)): matrix of 2-dimensioal word embeddings
            word2ind (dict): dictionary that maps word to indices for matrix M
            words (list of strings): words whose embeddings we want to visualize
    """

    # ------------------
    # Write your implementation here.
    
    
    # ------------------

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness.
# The plot produced should look like the included file question_1.4_test.png 
# ---------------------

print ("-" * 80)
print ("Outputted Plot:")

M_reduced_plot_test = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1], [0, 0]])
word2ind_plot_test = {'test1': 0, 'test2': 1, 'test3': 2, 'test4': 3, 'test5': 4}
words = ['test1', 'test2', 'test3', 'test4', 'test5']
plot_embeddings(M_reduced_plot_test, word2ind_plot_test, words)

print ("-" * 80)

### 问题 1.5：共现图分析 [书面回答]（3 分）

现在我们会把你写好的所有部分组合起来！我们会在 Large Movie Review 语料上使用固定窗口 4（默认窗口大小）计算共现矩阵。然后使用 TruncatedSVD 计算每个词的二维嵌入。TruncatedSVD 返回的是 U*S，因此我们需要对返回的向量进行归一化，让所有向量都出现在单位圆附近（因此接近性表示方向上的接近）。**注意**：下面进行归一化的代码行使用了 NumPy 的*广播*概念。如果你不了解广播，请查看
[Jake VanderPlas 的 Computation on Arrays: Broadcasting](https://jakevdp.github.io/PythonDataScienceHandbook/02.05-computation-on-arrays-broadcasting.html)。

运行下面的单元来生成图像。它可能需要几分钟才能运行完成。

In [ ]:
# -----------------------------
# Run This Cell to Produce Your Plot
# ------------------------------
imdb_corpus = read_corpus()
M_co_occurrence, word2ind_co_occurrence = compute_co_occurrence_matrix(imdb_corpus)
M_reduced_co_occurrence = reduce_to_k_dim(M_co_occurrence, k=2)

# Rescale (normalize) the rows to make them each of unit-length
M_lengths = np.linalg.norm(M_reduced_co_occurrence, axis=1)
M_normalized = M_reduced_co_occurrence / M_lengths[:, np.newaxis] # broadcasting

words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_normalized, word2ind_co_occurrence, words)

**请确认你的图像与作业压缩包中的 “question_1.5.png” 匹配。如果不匹配，请使用 “question_1.5.png” 中的图像来回答接下来的两个问题。**

a. 找出至少两组在二维嵌入空间中聚集在一起的词。请为你观察到的每个聚类给出解释。

#### <font color="red">在此处写下你的答案。</font>


b. 有哪些你认为应该聚在一起、但实际上没有聚在一起的词？请描述至少两个例子。

#### <font color="red">在此处写下你的答案。</font>

## 第 2 部分：基于预测的词向量（15 分）

正如课堂上讨论的那样，近年来基于预测的词向量展现出了更好的性能，例如 word2vec 和 GloVe（GloVe 也利用了计数的优势）。在这里，我们将探索 GloVe 生成的嵌入。关于 word2vec 和 GloVe 算法的更多细节，请重新阅读课堂笔记和课件。如果你想挑战一下自己，可以尝试阅读 [GloVe 的原始论文](https://nlp.stanford.edu/pubs/glove.pdf)。

然后运行以下单元，将 GloVe 向量加载到内存中。**注意**：如果这是你第一次运行这些单元，也就是需要下载嵌入模型，那么运行可能需要几分钟。如果你之前已经运行过这些单元，再次运行时会加载本地模型而不重新下载，通常需要约 1 到 2 分钟。

In [ ]:
def load_embedding_model():
    """ Load GloVe Vectors
        Return:
            wv_from_bin: All 400000 embeddings, each length 200
    """
    import gensim.downloader as api
    wv_from_bin = api.load("glove-wiki-gigaword-200")
    print("Loaded vocab size %i" % len(list(wv_from_bin.index_to_key)))
    return wv_from_bin
wv_from_bin = load_embedding_model()

#### 注意：如果你遇到 “reset by peer” 错误，请重新运行该单元以重新开始下载。

### 降低词嵌入的维度
让我们直接将 GloVe 嵌入与共现矩阵得到的嵌入进行比较。为了避免内存不足，我们会改用 40000 个 GloVe 向量的样本。
运行下面的单元以：

1. 将 40000 个 Glove 向量放入矩阵 M
2. 运行 `reduce_to_k_dim`（你的 Truncated SVD 函数），将向量从 200 维降到 2 维。

In [ ]:
def get_matrix_of_vectors(wv_from_bin, required_words):
    """ Put the GloVe vectors into a matrix M.
        Param:
            wv_from_bin: KeyedVectors object; the 400000 GloVe vectors loaded from file
        Return:
            M: numpy matrix shape (num words, 200) containing the vectors
            word2ind: dictionary mapping each word to its row number in M
    """
    import random
    words = list(wv_from_bin.index_to_key)
    print("Shuffling words ...")
    random.seed(225)
    random.shuffle(words)
    print("Putting %i words into word2ind and matrix M..." % len(words))
    word2ind = {}
    M = []
    curInd = 0
    for w in words:
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    for w in required_words:
        if w in words:
            continue
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    M = np.stack(M)
    print("Done.")
    return M, word2ind

In [ ]:
# -----------------------------------------------------------------
# Run Cell to Reduce 200-Dimensional Word Embeddings to k Dimensions
# Note: This should be quick to run
# -----------------------------------------------------------------
M, word2ind = get_matrix_of_vectors(wv_from_bin, words)
M_reduced = reduce_to_k_dim(M, k=2)

# Rescale (normalize) the rows to make them each of unit-length
M_lengths = np.linalg.norm(M_reduced, axis=1)
M_reduced_normalized = M_reduced / M_lengths[:, np.newaxis] # broadcasting

**注意：如果你在本地机器上遇到内存不足的问题，请尝试关闭其它应用程序以释放设备内存。你可能也想尝试重启机器，以便释放额外内存。然后立即运行 jupyter notebook，看看是否能正确加载词向量。如果这样之后你在本地机器加载嵌入时仍然有问题，请参加 office hours 或联系课程工作人员。**

### 问题 2.1：GloVe 图分析 [书面回答]（3 分）

运行下面的单元，为 `['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']` 绘制二维 GloVe 嵌入。

In [ ]:
words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_reduced_normalized, word2ind, words)

**请确认你的图像与作业压缩包中的 “question_2.1.png” 匹配。如果不匹配，请使用 “question_2.1.png” 中的图像（以及如果适用的话，“question_1.5.png” 中的图像）来回答接下来的两个问题。**

a. 这个图与之前由共现矩阵生成的图相比，有一个什么不同点？又有一个什么相似点？

#### <font color="red">在此处写下你的答案。</font>

b. 为什么 GloVe 图（question_2.1.png）可能会不同于之前由共现矩阵生成的图（question_1.5.png）？

#### <font color="red">在此处写下你的答案。</font>

### 余弦相似度
现在我们有了词向量，需要一种方式来根据这些向量量化单个词之间的相似性。其中一种度量是余弦相似度。我们将用它来寻找彼此“接近”和“远离”的词。

我们可以把 n 维向量看作 n 维空间中的点。如果从这个角度看，[L1](http://mathworld.wolfram.com/L1-Norm.html) 和 [L2](http://mathworld.wolfram.com/L2-Norm.html) 距离可以帮助量化从一个点到另一个点“必须走过”的空间距离。另一种方法是考察两个向量之间的夹角。根据三角学，我们知道：

<img src="./imgs/inner_product.png" width=20% style="float: center;"></img>

我们不必计算实际角度，而可以将相似度保留为 $similarity = cos(\Theta)$ 的形式。形式化地说，两个向量 $p$ 和 $q$ 之间的[余弦相似度](https://en.wikipedia.org/wiki/Cosine_similarity) $s$ 定义为：

$$s = \frac{p \cdot q}{||p|| ||q||}, \textrm{ where } s \in [-1, 1] $$ 

### 问题 2.2：具有多个含义的词（1.5 分）[代码 + 书面回答]
多义词和同形异义词是具有多个含义的词（参见这个 [wiki 页面](https://en.wikipedia.org/wiki/Polysemy)，了解多义词和同形异义词之间的区别）。找出一个具有*至少两个不同含义*的词，使得根据余弦相似度得到的前 10 个最相似词中，包含来自这*两个*含义的相关词。例如，“leaves” 的前 10 个结果中既有 “go_away” 的含义，也有 “a_structure_of_a_plant” 的含义；“scoop” 则既有 “handed_waffle_cone” 的含义，也有 “lowdown” 的含义。在找到合适词之前，你可能需要尝试几个多义词或同形异义词。

请说明你发现的词，以及前 10 个结果中出现的多个含义。为什么你认为许多你尝试过的多义词或同形异义词没有成功（即前 10 个最相似词只包含该词的**一个**含义）？

**注意**：你应该使用 `wv_from_bin.most_similar(word)` 函数来获得前 10 个最相似词。该函数会根据与给定词的余弦相似度，对词汇表中的所有其它词进行排序。如需进一步帮助，请查看 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.most_similar)__。

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.3：同义词与反义词（2 分）[代码 + 书面回答]

在考虑余弦相似度时，通常更方便的方式是考虑余弦距离，它就是 1 - 余弦相似度。

找出三个词 $(w_1,w_2,w_3)$，其中 $w_1$ 和 $w_2$ 是同义词，$w_1$ 和 $w_3$ 是反义词，但余弦距离满足：Cosine Distance $(w_1,w_3) <$ Cosine Distance $(w_1,w_2)$。

例如，$w_1$="happy" 与 $w_3$="sad" 的距离比与 $w_2$="cheerful" 的距离更近。请找出一个不同的、满足上述条件的例子。找到例子后，请给出一个可能的解释，说明为什么会出现这种反直觉结果。

这里你应该使用 `wv_from_bin.distance(w1, w2)` 函数来计算两个词之间的余弦距离。如需进一步帮助，请查看 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.distance)__。

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.4：使用词向量进行类比 [书面回答]（1.5 分）
词向量已被证明*有时*能够表现出解决类比问题的能力。

举个例子，对于类比 “man : grandfather :: woman : x”（读作：man 之于 grandfather，就像 woman 之于 x），x 是什么？

在下面的单元中，我们会展示如何使用 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar)__中的 `most_similar` 函数，通过词向量寻找 x。该函数会寻找与 `positive` 列表中的词最相似、并与 `negative` 列表中的词最不相似的词（同时会省略输入词，因为输入词通常本身就是最相似的；参见[这篇论文](https://www.aclweb.org/anthology/N18-2039.pdf)）。类比的答案会具有最高的余弦相似度（返回的数值最大）。

In [ ]:
# Run this cell to answer the analogy -- man : grandfather :: woman : x
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'grandfather'], negative=['man']))

设 $m$、$g$、$w$ 和 $x$ 分别表示 `man`、`grandfather`、`woman` 以及答案的词向量。只使用向量 $m$、$g$、$w$，以及向量算术运算符 $+$ 和 $-$，写出我们要与 $x$ 最大化余弦相似度的表达式。

提示：回想一下，词向量只是表示一个词的多维向量。你可以用任意位置画一个二维示例来辅助理解。相对于 `grandfather` 和答案，`man` 与 `woman` 会位于坐标平面的什么位置？

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.5：寻找类比 [代码 + 书面回答]（1.5 分）
a. 对于前面的例子，显然 “grandmother” 完成了这个类比。但请给出直观解释，说明为什么 `most_similar` 函数也会给出像 “granddaughter”、“daughter” 或 “mother” 这样的词？

#### <font color="red">在此处写下你的答案。</font>

b. 找出一个根据这些向量成立的类比例子（即目标词排名第一）。在你的解答中，请以 x:y :: a:b 的形式写出完整类比。如果你认为这个类比比较复杂，请用一两句话解释为什么该类比成立。

**注意**：你可能需要尝试许多类比，才能找到一个有效的例子！

In [ ]:
# For example: x, y, a, b = ("", "", "", "")
# ------------------
# Write your implementation here.


# ------------------

# Test the solution
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] == b

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.6：错误类比 [代码 + 书面回答]（1.5 分）
a. 在下面，我们期望看到的目标类比是 “hand : glove :: foot : **sock**”，但实际却出现了意外结果。请给出一个可能原因，解释为什么这个特定类比会得到这样的结果？

In [ ]:
pprint.pprint(wv_from_bin.most_similar(positive=['foot', 'glove'], negative=['hand']))

#### <font color="red">在此处写下你的答案。</font>

b. 再找一个根据这些向量并不成立的类比例子。在你的解答中，请以 x:y :: a:b 的形式写出预期类比，并说明根据词向量得到的 **错误** b 值（在前面的例子中，这个值会是 **'45,000-square'**）。

In [ ]:
# For example: x, y, a, b = ("", "", "", "")
# ------------------
# Write your implementation here.


# ------------------
pprint.pprint(wv_from_bin.most_similar(positive=[a, y], negative=[x]))
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] != b

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.7：词向量偏见的引导分析 [书面回答]（1 分）

意识到词嵌入中隐含的偏见（性别、种族、性取向等）非常重要。偏见可能很危险，因为它会通过使用这些模型的应用强化刻板印象。

运行下面的单元，考察：(a) 哪些词与 “man” 和 “profession” 最相似，同时与 “woman” 最不相似；以及 (b) 哪些词与 “woman” 和 “profession” 最相似，同时与 “man” 最不相似。指出女性相关词列表和男性相关词列表之间的差异，并解释它如何反映了性别偏见。

In [ ]:
# Run this cell
# Here `positive` indicates the list of words to be similar to and `negative` indicates the list of words to be
# most dissimilar from.

pprint.pprint(wv_from_bin.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'profession'], negative=['man']))

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.8：词向量偏见的独立分析 [代码 + 书面回答]（1 分）

使用 `most_similar` 函数找出另一组类比，展示这些向量体现出的某种偏见。请简要解释你发现的偏见示例。

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.9：思考偏见 [书面回答]（2 分）

a. 给出一个可能的解释，说明偏见如何进入词向量。你的解释应该聚焦于词向量，而不是其它 AI 系统（例如 ChatGPT）中的偏见。如有必要，你可以使用具体历史例子来支持你的解释。

#### <font color="red">在此处写下你的答案。</font>

b. 你可以使用什么方法来缓解词向量表现出的偏见？请简要解释该方法，以及该方法的目标是什么。


#### <font color="red">在此处写下你的答案。</font>

# <font color="blue">提交说明</font>

1. 点击 Jupyter Notebook 顶部的 Save 按钮。
2. 选择 Edit -> Clear Outputs of All Cells。这会清除所有单元的输出（但会保留所有单元的内容）。
2. 选择 Run -> Run All Cells。这会按顺序运行所有单元，并且需要几分钟。
3. 当你重新运行完所有内容后，选择 File -> Save and Export Notebook as -> PDF（如果你看到类似 <font color="red">“nbconvert failed: Pandoc wasn't found”</font> 的错误，可以先保存为 HTML）。选择 File -> Save and Export Notebook as -> HTML。这会将 notebook 保存为你电脑上的 HTML 文件。在浏览器中打开下载的 HTML 文件。在浏览器中按 Ctrl + P（Windows/Linux）或 Cmd + P（Mac）打开打印对话框。在打印对话框中，将目标位置改为 Save as PDF，然后点击 Save。<font color='blue'>请确保你的所有解答，尤其是代码部分，都显示在 pdf 中</font>；如果提供的代码因为代码单元中没有自动换行而被截断，这是可以接受的。
4. 查看 PDF 文件，确保你的所有解答都在其中并且显示正确。PDF 是评分人员唯一会看到的内容！
5. 在 Gradescope 上提交你的 PDF。